# Minggu 1 (versi perbaikan) — sesuai draf Topik 3

Cocokkan dengan jadwal draf minggu 1:
- **H1** unduh ZIP + CSV Bug Bytes dan CSV getaran USDA
- **H2–3** pilih file hama gudang dan noise, cek fs, resample
- **H4–5** skrip highpass, RMS, dan deteksi event

Yang diperbaiki dari `01_minggu1.ipynb`:
1. File noise dan CSV USDA ikut dicek (sebelumnya belum ada)
2. Resample benar-benar dijalankan ke satu fs bersama dan hasilnya disimpan
3. Highpass 200 Hz ditambahkan (sebelumnya hanya buang rata-rata/DC)
4. Urutan: baca → resample → highpass → RMS 10 ms → event
5. Interval antar event dihitung, dan semua file diproses (sebelumnya hanya A5)
6. `savefig` dipindah ke sel yang sama dengan gambarnya (sebelumnya menyimpan gambar kosong)
7. Spektrogram pakai Hann 1024 seperti di draf

In [1]:
import glob, os
from fractions import Fraction
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy.signal import butter, sosfiltfilt, resample_poly, spectrogram

# Penentuan path fleksibel (dapat dijalankan dari root maupun dari subfolder notebooks)
if os.path.exists("../data/raw"):
    RAW  = "../data/raw"
    PROC = "../data/processed"
    RES  = "../results"
elif os.path.exists("Sinyal-Akustik-Kutu-Beras-/data/raw"):
    RAW  = "Sinyal-Akustik-Kutu-Beras-/data/raw"
    PROC = "Sinyal-Akustik-Kutu-Beras-/data/processed"
    RES  = "Sinyal-Akustik-Kutu-Beras-/results"
else:
    RAW  = "data/raw"
    PROC = "data/processed"
    RES  = "results"

os.makedirs(PROC, exist_ok=True)
os.makedirs(RES, exist_ok=True)

# ---- parameter sesuai draf ----
FS_BERSAMA = 16000   # fs bersama: KEPUTUSAN TIM (harus <= fs terkecil file yang dipakai)
HP_HZ      = 200     # highpass 200 Hz
FRAME_S    = 0.01    # RMS per frame 10 ms
K_MAD      = 3       # ambang = median + 3*MAD
print("siap")


siap


## H2–3 · Inventaris file (hama + noise) dan cek fs

Kolom `peran` diisi lewat kamus `META`. Kode file yang belum ada di `META` akan tampil sebagai
"belum dipilih". **Untuk noise:** unduh file noise Bug Bytes ke `../data/raw`, jalankan sel ini sekali
untuk melihat namanya, lalu tambahkan barisnya di `META` (contoh ada di komentar) dan jalankan ulang.

In [2]:
# kode -> (peran, spesies, sensor)
META = {
    # File hama (S. oryzae) — 5 variasi sensor sesuai draf
    "A3": ("hama", "S. oryzae (larva)", "PVDF"),
    "A4": ("hama", "S. oryzae (larva)", "akselerometer"),
    "A5": ("hama", "S. oryzae (larva)", "ultrasonik 40 kHz"),
    "A6": ("hama", "S. oryzae (larva)", "ultrasonik 30 kHz"),
    "A7": ("hama", "S. oryzae (larva)", "piezoelektrik"),
    # File hama (S. zeamais) — larva dan dewasa
    "A8": ("hama", "S. zeamais (larva)", "mikrofon"),
    "A9": ("hama", "S. zeamais (dewasa)", "mikrofon"),
    # File noise lapangan Bug Bytes (Kategori I: environmental noise)
    "I3": ("noise", "background", "akselerometer (derau pesawat)"),
    "I4": ("noise", "background", "akselerometer (derau truk dan pesawat)"),
    "I6": ("noise", "background", "akselerometer (derau angin lapangan)"),
}

baris = []
for path in sorted(glob.glob(f"{RAW}/*.wav")):
    nama = os.path.basename(path)[:-4]
    kode = nama.split("-")[0]
    fs, x = wavfile.read(path)
    peran, spesies, sensor = META.get(kode, ("belum dipilih", "?", "?"))
    baris.append({"kode": kode, "file": nama + ".wav", "peran": peran,
                  "spesies": spesies, "sensor": sensor,
                  "fs_hz": fs, "durasi_s": round(len(x) / fs, 1),
                  "kanal": 1 if x.ndim == 1 else x.shape[1], "dtype": str(x.dtype)})
inv = pd.DataFrame(baris)
inv.to_csv(f"{RES}/inventaris.csv", index=False)
inv


Out[0]: 
   kode                       file          peran  ... durasi_s kanal  dtype
0    A1            A1-dfaccs11.wav  belum dipilih  ...     30.0     1  int16
1   A10    A10-ptr-l-m4r2f4-2s.wav  belum dipilih  ...     18.6     1  int16
2   A11       A11-ptr_adu1-72s.wav  belum dipilih  ...     21.0     1  int16
3   A12  A12-roachmoves_m54524.wav  belum dipilih  ...     35.4     1  int16
4    A2            A2-piezimm1.wav  belum dipilih  ...     10.0     1  int16
..  ...                        ...            ...  ...      ...   ...    ...
90   I3            I3-monrbck1.wav          noise  ...     15.0     1  int16
91   I4            I4-monrbck2.wav          noise  ...     15.0     1  int16
92   I5            I5-monrbck3.wav  belum dipilih  ...     15.0     1  int16
93   I6              I6-nwrec1.wav          noise  ...     15.0     1  int16
94   I7             I7-dmnurs1.wav  belum dipilih  ...     15.0     1  int16

[95 rows x 9 columns]


   kode                       file          peran  ... durasi_s kanal  dtype
0    A1            A1-dfaccs11.wav  belum dipilih  ...     30.0     1  int16
1   A10    A10-ptr-l-m4r2f4-2s.wav  belum dipilih  ...     18.6     1  int16
2   A11       A11-ptr_adu1-72s.wav  belum dipilih  ...     21.0     1  int16
3   A12  A12-roachmoves_m54524.wav  belum dipilih  ...     35.4     1  int16
4    A2            A2-piezimm1.wav  belum dipilih  ...     10.0     1  int16
..  ...                        ...            ...  ...      ...   ...    ...
90   I3            I3-monrbck1.wav          noise  ...     15.0     1  int16
91   I4            I4-monrbck2.wav          noise  ...     15.0     1  int16
92   I5            I5-monrbck3.wav  belum dipilih  ...     15.0     1  int16
93   I6              I6-nwrec1.wav          noise  ...     15.0     1  int16
94   I7             I7-dmnurs1.wav  belum dipilih  ...     15.0     1  int16

[95 rows x 9 columns]

In [3]:
# cek peran yang masih kurang
print("jumlah file per peran:")
print(inv["peran"].value_counts().to_string())
if (inv["peran"] == "noise").sum() == 0:
    print("\n[!] Belum ada file NOISE yang dipilih (draf: 'pilih file hama gudang dan file noise').")

fs_dipakai = inv.loc[inv["peran"].isin(["hama", "noise"]), "fs_hz"]
print("fs terkecil file terpilih:", fs_dipakai.min(), "Hz | FS_BERSAMA:", FS_BERSAMA, "Hz")
assert FS_BERSAMA <= fs_dipakai.min(), "FS_BERSAMA lebih besar dari fs terkecil: akan terjadi upsampling"
assert HP_HZ < FS_BERSAMA / 2, "highpass harus di bawah Nyquist"

jumlah file per peran:
peran
belum dipilih    85
hama              7
noise             3
fs terkecil file terpilih: 20000 Hz | FS_BERSAMA: 16000 Hz


## H1 · Cek CSV (inventaris Bug Bytes dan getaran USDA)

Letakkan CSV-nya di `../data/raw`. Sel ini hanya mengecek bahwa file terbaca dan melihat kolomnya.

In [4]:
csvs = sorted(glob.glob(f"{RAW}/*.csv"))
if not csvs:
    print("[!] Belum ada CSV di", RAW, "(butuh: CSV inventaris Bug Bytes + CSV getaran USDA)")
for p in csvs:
    d = pd.read_csv(p)
    print("=====", os.path.basename(p), d.shape)
    print("kolom:", list(d.columns))
    display(d.head(3))

===== Beetle Vibration Data.csv (159, 12)
kolom: ['Unnamed: 0', 'Trial', 'Species', 'Density', 'Av_Int', 'Av_Dur', 'Count', 'Freq', 'ID', 'logInt', 'logCount', 'logDur']
   Unnamed: 0 Trial       Species  ...    logInt  logCount    logDur
0           2     A   R. dominica  ... -0.060104  1.690196 -2.728679
1           3     A   R. dominica  ... -2.881006  0.778151 -2.803492
2           4     A  T. castaneum  ... -0.932548  2.318063 -2.548963

[3 rows x 12 columns]
===== Insect-Pest-Sounds-inventory_1.csv (95, 3)
kolom: ['Category', 'File ID', 'Description']
                                            Category  ...                                        Description
0  A.  Stored Product Insect movement and feeding...  ...  Plodia interpunctella larvae in dry dog food [...
1  A.  Stored Product Insect movement and feeding...  ...  Individual Plodia interpunctella larva in dry ...
2  A.  Stored Product Insect movement and feeding...  ...  Sitophilus oryzae larvae (17-18 d old) in whea...


## H4–5 · Fungsi praproses: resample → highpass → RMS → event

In [5]:
def baca_wav(path):
    fs, x = wavfile.read(path)
    x = x.astype(float)
    if x.ndim > 1:                 # kalau stereo, ratakan jadi mono
        x = x.mean(axis=1)
    return fs, x - x.mean()        # buang offset DC

def resample_ke(x, fs, fs_baru):
    if fs == fs_baru:
        return x
    r = Fraction(fs_baru, fs)
    return resample_poly(x, r.numerator, r.denominator)   # sudah ada filter anti-aliasing

def highpass(x, fs, fc=HP_HZ, order=4):
    sos = butter(order, fc, btype="highpass", fs=fs, output="sos")
    return sosfiltfilt(sos, x)     # dua arah, jadi tidak menggeser waktu event

def rms_frame(x, fs, frame_s=FRAME_S):
    n = int(round(fs * frame_s))
    m = len(x) // n
    frames = x[:m * n].reshape(m, n)
    return np.sqrt((frames ** 2).mean(axis=1))

def deteksi_event(rms, frame_s=FRAME_S, k=K_MAD):
    med = np.median(rms)
    mad = np.median(np.abs(rms - med))
    ambang = med + k * mad
    aktif = rms > ambang
    d = np.diff(np.concatenate([[0], aktif.astype(int), [0]]))
    mulai = np.where(d == 1)[0]
    akhir = np.where(d == -1)[0]
    ev = pd.DataFrame({"mulai_s": mulai * frame_s, "akhir_s": akhir * frame_s})
    ev["durasi_ms"] = (ev.akhir_s - ev.mulai_s) * 1000
    ev["jeda_ke_berikut_s"] = ev.mulai_s.shift(-1) - ev.akhir_s       # akhir -> mulai berikutnya
    ev["interval_mulai_s"]  = ev.mulai_s.shift(-1) - ev.mulai_s       # mulai -> mulai berikutnya
    return ev, ambang

## Jalankan untuk semua file terpilih (bukan hanya A5)

Hasil: sinyal terproses disimpan di `../data/processed`, tabel event per file di `../results`.
File noise hanya di-resample + highpass (tanpa deteksi event).

In [6]:
hasil, ringkas = {}, []
for path in sorted(glob.glob(f"{RAW}/*.wav")):
    nama = os.path.basename(path)[:-4]
    kode = nama.split("-")[0]
    peran, spesies, sensor = META.get(kode, ("belum dipilih", "?", "?"))
    if peran not in ("hama", "noise"):
        continue

    fs, x = baca_wav(path)
    y = resample_ke(x, fs, FS_BERSAMA)
    y = highpass(y, FS_BERSAMA)
    wavfile.write(f"{PROC}/{kode}_{FS_BERSAMA // 1000}k_hp{HP_HZ}.wav", FS_BERSAMA, y.astype(np.float32))

    if peran == "noise":
        ringkas.append({"kode": kode, "peran": peran, "fs_asal": fs, "durasi_s": round(len(y) / FS_BERSAMA, 1)})
        continue

    rms = rms_frame(y, FS_BERSAMA)
    ev, ambang = deteksi_event(rms)
    ev.to_csv(f"{RES}/event_{kode}.csv", index=False)
    hasil[kode] = dict(y=y, rms=rms, ev=ev, ambang=ambang, fs_asal=fs, x_asli=x)
    ringkas.append({"kode": kode, "peran": peran, "spesies": spesies, "sensor": sensor,
                    "fs_asal": fs, "durasi_s": round(len(y) / FS_BERSAMA, 1),
                    "n_event": len(ev), "ambang": round(ambang, 2),
                    "median_durasi_ms": round(ev.durasi_ms.median(), 1) if len(ev) else None,
                    "median_jeda_s": round(ev.jeda_ke_berikut_s.median(), 3) if len(ev) > 1 else None})

ringkasan = pd.DataFrame(ringkas)
ringkasan.to_csv(f"{RES}/ringkasan_praproses.csv", index=False)
ringkasan

Out[0]: 
  kode  peran              spesies  ...   ambang  median_durasi_ms  median_jeda_s
0   A3   hama    S. oryzae (larva)  ...   444.90              10.0          0.070
1   A4   hama    S. oryzae (larva)  ...   512.99              10.0          0.050
2   A5   hama    S. oryzae (larva)  ...  1358.76              10.0          0.120
3   A6   hama    S. oryzae (larva)  ...  2272.89              10.0          0.245
4   A7   hama    S. oryzae (larva)  ...   941.72              10.0          0.050
5   A8   hama   S. zeamais (larva)  ...  1375.35              10.0          0.100
6   A9   hama  S. zeamais (dewasa)  ...   429.01              10.0          0.050
7   I3  noise                  NaN  ...      NaN               NaN            NaN
8   I4  noise                  NaN  ...      NaN               NaN            NaN
9   I6  noise                  NaN  ...      NaN               NaN            NaN

[10 rows x 10 columns]


  kode  peran              spesies  ...   ambang  median_durasi_ms  median_jeda_s
0   A3   hama    S. oryzae (larva)  ...   444.90              10.0          0.070
1   A4   hama    S. oryzae (larva)  ...   512.99              10.0          0.050
2   A5   hama    S. oryzae (larva)  ...  1358.76              10.0          0.120
3   A6   hama    S. oryzae (larva)  ...  2272.89              10.0          0.245
4   A7   hama    S. oryzae (larva)  ...   941.72              10.0          0.050
5   A8   hama   S. zeamais (larva)  ...  1375.35              10.0          0.100
6   A9   hama  S. zeamais (dewasa)  ...   429.01              10.0          0.050
7   I3  noise                  NaN  ...      NaN               NaN            NaN
8   I4  noise                  NaN  ...      NaN               NaN            NaN
9   I6  noise                  NaN  ...      NaN               NaN            NaN

[10 rows x 10 columns]

## Cek highpass: frekuensi puncak sebelum vs sesudah

Di notebook lama, A4 (akselerometer) puncaknya 0 Hz, artinya yang dominan itu gerakan sangat lambat/DC, bukan bunyi serangga.
Kolom terakhir menunjukkan berapa persen energi yang berada di bawah 200 Hz (yang akan dibuang highpass).

In [7]:
def puncak_hz(x, fs):
    f = np.fft.rfftfreq(len(x), 1 / fs)
    P = np.abs(np.fft.rfft(x)) ** 2
    return float(f[np.argmax(P)])

def pct_energi_bawah(x, fs, fc=HP_HZ):
    f = np.fft.rfftfreq(len(x), 1 / fs)
    P = np.abs(np.fft.rfft(x)) ** 2
    return 100 * P[f < fc].sum() / P.sum()

cek = []
for kode, h in hasil.items():
    cek.append({"kode": kode,
                "puncak_sebelum_Hz": round(puncak_hz(h["x_asli"], h["fs_asal"])),
                "puncak_sesudah_Hz": round(puncak_hz(h["y"], FS_BERSAMA)),
                "energi_di_bawah_200Hz_%": round(pct_energi_bawah(h["x_asli"], h["fs_asal"]), 1)})
pd.DataFrame(cek)

Out[0]: 
  kode  puncak_sebelum_Hz  puncak_sesudah_Hz  energi_di_bawah_200Hz_%
0   A3               5183               5183                      0.8
1   A4                  0               1185                      5.6
2   A5               5179               5179                      0.5
3   A6               5185               5185                      0.3
4   A7               2242               2242                      0.0
5   A8                243                243                      0.7
6   A9                257                554                      0.4


  kode  puncak_sebelum_Hz  puncak_sesudah_Hz  energi_di_bawah_200Hz_%
0   A3               5183               5183                      0.8
1   A4                  0               1185                      5.6
2   A5               5179               5179                      0.5
3   A6               5185               5185                      0.3
4   A7               2242               2242                      0.0
5   A8                243                243                      0.7
6   A9                257                554                      0.4

## Gambar ringkas per file (disimpan SEBELUM `plt.show()`)

Panel: sinyal, spektrum (FFT), spektrogram (Hann, 1024), RMS + ambang + awal event.

In [8]:
def plot_ringkas(kode):
    h = hasil[kode]; y = h["y"]; rms = h["rms"]; ev = h["ev"]; amb = h["ambang"]
    t = np.arange(len(y)) / FS_BERSAMA
    fig, ax = plt.subplots(4, 1, figsize=(12, 13))

    ax[0].plot(t, y, lw=0.5)
    ax[0].set(title=f"{kode}: sinyal setelah resample {FS_BERSAMA} Hz + highpass {HP_HZ} Hz",
              xlabel="detik", ylabel="amplitudo")

    f = np.fft.rfftfreq(len(y), 1 / FS_BERSAMA)
    ax[1].semilogy(f, np.abs(np.fft.rfft(y)) ** 2 + 1e-12, lw=0.6)
    ax[1].set(title="spektrum (FFT)", xlabel="Hz", ylabel="energi")

    ff, tt, S = spectrogram(y, FS_BERSAMA, window="hann", nperseg=1024, noverlap=512)
    im = ax[2].pcolormesh(tt, ff, 10 * np.log10(S + 1e-12), shading="auto")
    ax[2].set(title="spektrogram (Hann, 1024)", xlabel="detik", ylabel="Hz")
    fig.colorbar(im, ax=ax[2], label="dB")

    tf = np.arange(len(rms)) * FRAME_S
    ax[3].plot(tf, rms)
    ax[3].axhline(amb, color="r", label="ambang median+3 MAD")
    ax[3].scatter(ev.mulai_s, [amb] * len(ev), color="g", s=12, zorder=3, label="awal event")
    ax[3].set(title=f"RMS 10 ms, {len(ev)} event", xlabel="detik", ylabel="RMS")
    ax[3].legend()

    fig.tight_layout()
    fig.savefig(f"{RES}/ringkas_{kode}.png", dpi=150)   # simpan dulu...
    plt.show()                                          # ...baru tampilkan
    plt.close(fig)

for kode in hasil:
    plot_ringkas(kode)

<ipython-input-1-49aabc1f53b9>:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()                                          # ...baru tampilkan


## Checklist akhir minggu 1 (otomatis)

In [9]:
def tanda(ok): return "✓" if ok else "✗"

hama = [k for k, v in META.items() if v[0] == "hama"]
noise = [k for k, v in META.items() if v[0] == "noise"]
ada = set(inv["kode"])

print(tanda(len(csvs) >= 2),  "CSV terunduh (inventaris Bug Bytes + getaran USDA) — ditemukan:", len(csvs))
print(tanda(all(k in ada for k in hama)), "file hama terpilih ada di folder dan fs tercatat")
print(tanda(len(noise) > 0 and all(k in ada for k in noise)), "file noise terpilih")
print(tanda(os.path.exists(f"{RES}/inventaris.csv")), "inventaris.csv tersimpan")
print(tanda(all(os.path.exists(f"{PROC}/{k}_{FS_BERSAMA // 1000}k_hp{HP_HZ}.wav") for k in hama)),
      f"semua file hama sudah resample {FS_BERSAMA} Hz + highpass {HP_HZ} Hz")
print(tanda(all(os.path.exists(f"{RES}/event_{k}.csv") for k in hama)), "tabel event ada untuk semua file hama")

✓ CSV terunduh (inventaris Bug Bytes + getaran USDA) — ditemukan: 2
✓ file hama terpilih ada di folder dan fs tercatat
✓ file noise terpilih
✓ inventaris.csv tersimpan
✓ semua file hama sudah resample 16000 Hz + highpass 200 Hz
✓ tabel event ada untuk semua file hama
